In [ ]:
from dotenv import load_dotenv
import plotly.io as pio
import neptune

load_dotenv('.tokens.env', override=True)

project = neptune.init_project(project='mtyrol/drop')
project

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/mtyrol/drop/


# Sokoban

In [9]:
df = project.fetch_runs_table(tag=['inner_eval', 'sokoban']).to_pandas()

In [10]:
df

,sys/creation_time,sys/description,sys/failed,sys/group_tags,sys/hostname,sys/id,sys/modification_time,sys/monitoring_time,sys/name,sys/owner,...,monitoring/9d72c8e6/cpu,monitoring/9d72c8e6/gpu_memory,monitoring/9d72c8e6/gpu_power,monitoring/9d72c8e6/hostname,monitoring/9d72c8e6/memory,monitoring/9d72c8e6/pid,monitoring/9d72c8e6/stderr,monitoring/9d72c8e6/stdout,monitoring/9d72c8e6/tid,monitoring/9d72c8e6/traceback
0,2025-06-29 02:49:28.964,adasubs solve,True,,DESKTOP-4QFGA24,DROP-628,2025-06-29 12:53:51.607,36261,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-29 01:46:14.276,adasubs solve,False,,DESKTOP-4QFGA24,DROP-627,2025-06-29 02:49:28.616,3794,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-06-28 18:17:17.642,adasubs solve,True,,DESKTOP-4QFGA24,DROP-616,2025-06-28 21:40:46.022,12208,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-06-28 07:03:10.869,adasubs solve,True,,DESKTOP-4QFGA24,DROP-600,2025-06-28 10:39:18.913,12968,adasubs solve,mtyrol,...,100.0,0.12793,3.1,DESKTOP-4QFGA24,8.658340,5048,\n,[neptune] [info ] Shutting down background j...,131576802465600,
4,2025-06-28 02:44:43.310,adasubs solve,False,,DESKTOP-4QFGA24,DROP-599,2025-06-28 07:03:10.175,15506,adasubs solve,mtyrol,...,25.3,0.12793,3.1,DESKTOP-4QFGA24,6.569489,5048,NaN,[neptune] [info ] Shutting down background j...,131576802465600,NaN
5,2025-06-28 01:48:09.084,adasubs solve,False,,DESKTOP-4QFGA24,DROP-598,2025-06-28 02:44:42.830,3394,adasubs solve,mtyrol,...,48.1,0.12793,3.1,DESKTOP-4QFGA24,7.750130,5048,[neptune] [warning] /home/mtyrolski/.pyenv/ver...,[neptune] [info ] Shutting down background j...,131576802465600,NaN
6,2025-06-25 19:31:20.151,adasubs solve,False,,DESKTOP-4QFGA24,DROP-595,2025-06-25 21:01:56.171,5436,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-06-25 17:09:05.404,adasubs solve,False,,DESKTOP-4QFGA24,DROP-594,2025-06-25 19:31:19.129,8534,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-06-25 15:30:06.028,adasubs solve,False,,DESKTOP-4QFGA24,DROP-593,2025-06-25 17:09:04.566,5938,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-06-25 14:50:25.636,adasubs solve,False,,DESKTOP-4QFGA24,DROP-592,2025-06-25 15:30:05.066,2380,adasubs solve,mtyrol,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences
ALL_BUDGET_VALUES = [1, 2, 5, 10, 25, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 250, 300, 350, 400, 450, 500, 600, 800, 1000]


from dataclasses import dataclass
from itertools import product
from typing import Collection
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd
SYSID = 'sys/id'

@dataclass
class NeptuneParam:
    full_name_on_neptune: str
    short_name: str
    
params = list(map(
    lambda x: NeptuneParam(
        full_name_on_neptune=x,
        short_name=x.split('/')[-1]
    ),
    [
        'parameters/subgoal_generator/subgoal_generation_kwargs/num_return_sequences',
        'parameters/subgoal_generator/generator_k_list',
        *[f'solved/rate/{n}_nodes' for n in ALL_BUDGET_VALUES]
    ]
))

params_astar = list(map(
    lambda x: NeptuneParam(
        full_name_on_neptune=x,
        short_name=x.split('/')[-1]
    ),
    [
        'parameters/algorithm/solver/planner_class/depth_weight',
        'parameters/algorithm/solver/planner_class/value_weight',
        *[f'solved/rate/{n}_nodes' for n in ALL_BUDGET_VALUES]
    ]
))

from abc import ABC, abstractmethod

class PlotFbn(ABC):
    @abstractmethod
    def __call__(self,
                 runs_table: pd.DataFrame,
                 grouping_params: list[NeptuneParam],
                 values_params: list[NeptuneParam]) -> go.Figure:
        pass
    

class PlotSuccessRateByBudget(PlotFbn):
    def __call__(self,
                 runs_table: pd.DataFrame,
                 grouping_params: list[NeptuneParam],
                 values_params: list[NeptuneParam]) -> go.Figure:
        grouping_col_names = [param.full_name_on_neptune for param in grouping_params]
        unique_values = {
            param.short_name: runs_table[param.full_name_on_neptune].unique()
            for param in grouping_params
        }
        
        all_combinations = runs_table[grouping_col_names].drop_duplicates().apply(
            lambda row: tuple(row),
            axis=1
        ).tolist()
        assert len(all_combinations) == len(set(all_combinations)), f"There should be no duplicate " \
            f"combinations of grouping parameters {len(all_combinations)} != {len(set(all_combinations))}"
        all_combinations = set(all_combinations)
        
        filtered_runs = runs_table[
            runs_table[grouping_col_names].apply(
                lambda row: tuple(row) in all_combinations,
                axis=1
            )
        ]
        
        # assert only one run per combination
        if len(filtered_runs) != len(all_combinations):
            # Find combinations with 2 or more runs
            combo_counts = runs_table.groupby(grouping_col_names).size().reset_index(name='count')
            multi_run_combos = combo_counts[combo_counts['count'] >= 2]
            details = []
            for _, row in multi_run_combos.iterrows():
                combo_filter = (runs_table[grouping_col_names] == row[grouping_col_names]).all(axis=1)
                run_names = runs_table.loc[combo_filter, SYSID].tolist()  
                combo = {col: row[col] for col in grouping_col_names}
                count = row['count']
                details.append(
                    f"Combination {combo} has {count} runs: {run_names}"
                )
            if details:
                details_str = "\n".join(details)
            else:
                details_str = "No combinations with >= 2 runs found."
            raise Exception(
                f"There should be one run per combination of grouping parameters "
                f"{len(filtered_runs)} != {len(all_combinations)}.\n"
                f"Multiple runs found for combinations:\n{details_str}"
            )
            
        values_params_filtered = [
            param for param in values_params
            if param.full_name_on_neptune in filtered_runs.columns
        ]    

        filtered_runs = filtered_runs[grouping_col_names + [param.full_name_on_neptune for param in values_params_filtered]]
        # rename to short names
        filtered_runs = filtered_runs.rename(
            columns={param.full_name_on_neptune: param.short_name for param in grouping_params + values_params_filtered}
        )
        
        fig = go.Figure()
        
        for group in all_combinations:
            group_filter = (filtered_runs[grouping_params[0].short_name] == group[0]) & \
                           (filtered_runs[grouping_params[1].short_name] == group[1])
            group_data = filtered_runs[group_filter]
            
            success_rate_list = [group_data[param.short_name].values[0] for param in values_params_filtered]
            fig.add_trace(
                go.Scatter(
                    x=ALL_BUDGET_VALUES,
                    y=success_rate_list,
                    mode='lines+markers',
                    name=f"{grouping_params[0].short_name}={group[0]}, {grouping_params[1].short_name}={group[1]}"
                )
            )
        fig.update_layout(
            title='Success Rate by Budget',
            xaxis_title='Budget (number of nodes)',
            yaxis_title='Success Rate',
            legend_title='Group'
        )
        
        fig.update_xaxes(tickvals=ALL_BUDGET_VALUES, ticktext=[str(b) for b in ALL_BUDGET_VALUES])
        fig.update_yaxes(tickformat=".0%")
        pio.templates.default = "plotly_white"
        fig.update_layout(template=pio.templates.default)
        fig.update_layout(width=1000, height=600)
        fig.update_layout(margin=dict(l=20, r=20, t=50, b=20))
        
        return fig

PlotSuccessRateByBudget()(runs_table=df,
                          grouping_params=params[:2],
                          values_params=params[2:])

In [15]:
df_astar_sokoban_grid = project.fetch_runs_table(tag=['sokoban', 'astar']).to_pandas()

In [16]:
PlotSuccessRateByBudget()(runs_table=df_astar_sokoban_grid,
                          grouping_params=params_astar[:2],
                          values_params=params_astar[2:])

# N-Puzzle

In [17]:
df_adasubs_npuzzle_grid = project.fetch_runs_table(tag=['npuzzle', 'inner_eval']).to_pandas()
df_astar_npuzzle_grid = project.fetch_runs_table(tag=['npuzzle', 'astar']).to_pandas()

In [18]:
PlotSuccessRateByBudget()(runs_table=df_adasubs_npuzzle_grid,
                            grouping_params=params[:2],
                            values_params=params[2:]).show()


In [19]:
PlotSuccessRateByBudget()(runs_table=df_astar_npuzzle_grid,
                            grouping_params=params_astar[:2],
                            values_params=params_astar[2:]).show()